In [ ]:
import pandas as pd

# Read the CSV and Perform Basic Data Cleaning

In [ ]:
df = pd.read_csv("cumulative.csv")
df = df.drop(columns=["rowid", "kepid", "kepoi_name", "kepler_name", "koi_pdisposition", "koi_score", "koi_tce_delivname"])
# Drop the null columns where all values are null
df = df.dropna(axis='columns', how='all')
# Drop the null rows
df = df.dropna()
df.head()

# Create a Train Test Split

Use `koi_disposition` for the y values

In [ ]:
# Use train_test_split to create training and testing data
from sklearn.model_selection import train_test_split
y = df['koi_disposition'].values.reshape(-1, 1)
data = df.drop(['koi_disposition'] ,axis=1)
feature_names = data.columns
data.head()

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(data, y, random_state=42)
X_train.head()

In [ ]:
# Pre-processing

#Scale the data using the MinMaxScaler
from sklearn.preprocessing import MinMaxScaler
X_minmax = MinMaxScaler().fit(X_train)

X_train_scaled = X_minmax.transform(X_train)
X_test_scaled = X_minmax.transform(X_test)

# Train the Support Vector Machine

In [ ]:
import numpy as np
# Support vector machine linear classifier
from sklearn.svm import SVC 
model = SVC(kernel='linear')
model.fit(X_train_scaled,np.ravel(y_train))

In [ ]:
print(f"Training Data Score: {model.score(X_train_scaled, y_train)}")
print(f"Testing Data Score: {model.score(X_test_scaled, y_test)}")

In [ ]:
model2 = SVC(kernel='linear')
model2.fit(X_test_scaled,np.ravel(y_test))

In [ ]:

print(f"Testing Data Score: {model2.score(X_test_scaled, y_test)}")

# Hyperparameter Tuning

Use `GridSearchCV` to tune the `C` and `gamma` parameters

In [ ]:
# Create the GridSearchCV model
from sklearn.model_selection import GridSearchCV
param_grid = {'C': [50,100, 300,500,800],
              'gamma': [1,0.1, 0.01,0.001,0.0001]}
grid = GridSearchCV(model, param_grid, verbose=3)

In [ ]:
# Train the model with GridSearch
grid.fit(X_train_scaled,np.ravel(y_train))


In [ ]:
print(grid.best_params_)
print(grid.best_score_)

In [ ]:
# Create the GridSearchCV model
from sklearn.model_selection import GridSearchCV
param_grid = {'C': [2000],
              'gamma': [1, 0.01]}
grid = GridSearchCV(model, param_grid, verbose=3)
grid.fit(X_train_scaled,np.ravel(y_train))

In [ ]:
print(grid.best_params_)
print(grid.best_score_)

# RANDOM FOREST CLASSIFIER

In [ ]:
#Random First Model
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=500)
rf = rf.fit(X_train_scaled, np.ravel(y_train))
rf.score(X_test_scaled,np.ravel(y_test))

In [ ]:
#Heatmap to see the correlations for the features
import matplotlib.pyplot as plt
import seaborn as sns
corr = data.corr()
# plot the heatmap
fig, ax = plt.subplots(figsize=(10,10)) 
sns.heatmap(corr,
        xticklabels=corr.columns,
        yticklabels=corr.columns, ax=ax)

In [ ]:
data1 = data.drop(["koi_time0bk_err2"],axis=1)
data1.head()

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(data1, y, random_state=42)
#Random First Model
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=500)
rf = rf.fit(X_train_scaled, np.ravel(y_train))
rf.score(X_test_scaled,np.ravel(y_test))

# K-Nearest Neighbour

In [ ]:
from sklearn.datasets import load_iris
from sklearn.neighbors import KNeighborsClassifier

In [ ]:
# Loop through different k values to see which has the highest accuracy
# Note: We only use odd numbers because we don't want any ties
train_scores = []
test_scores = []
for k in range(1, 20, 2):
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, np.ravel(y_train))
    train_score = knn.score(X_train_scaled, np.ravel(y_train))
    test_score = knn.score(X_test_scaled, np.ravel(y_test))
    train_scores.append(train_score)
    test_scores.append(test_score)
    print(f"k: {k}, Train/Test Score: {train_score:.3f}/{test_score:.3f}")
    
    
plt.plot(range(1, 20, 2), train_scores, marker='o')
plt.plot(range(1, 20, 2), test_scores, marker="x")
plt.xlabel("k neighbors")
plt.ylabel("Testing accuracy Score")
plt.show()

In [ ]:
# Note that k: 5 provides the best accuracy where the classifier starts to stablize
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, np.ravel(y_train))
print('k=5, Test Acc: %.3f' % knn.score(X_test_scaled, y_test))